<a href="https://colab.research.google.com/github/satyamkarn100-ctrl/Neural-Recommendation-Personalization-Engine/blob/main/temporal_split.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
from huggingface_hub import HfApi
from huggingface_hub import login
login()


In [2]:
!pip install -U huggingface_hub

In [3]:
file_path = hf_hub_download(
    repo_id="Satyamkarn100/NeuraRec-Electronics",
    filename="processed/reviews_clean.parquet",
    repo_type="dataset"
)
file_path0 = hf_hub_download(
    repo_id =  "Satyamkarn100/NeuraRec-Electronics",
    filename="processed/meta_clean.parquet",
    repo_type="dataset"
)

reviews = pd.read_parquet(file_path)
meta = pd.read_parquet(file_path0)

In [4]:
print('Shape of Reviews:',reviews.shape)
print("Shape of Meta:",meta.shape)
print("\ncolumns of Reviews",reviews.columns)

Shape of Reviews: (15473536, 7)
Shape of Meta: (1610012, 13)

columns of Reviews Index(['user_id', 'parent_asin', 'rating', 'timestamp', 'datetime', 'user_idx',
       'item_idx'],
      dtype='object')


In [5]:
reviews.head()

,user_id,parent_asin,rating,timestamp,datetime,user_idx,item_idx
0,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B0047T79VS,3.0,1344406083000,2012-08-08 06:08:03,928257,42467
1,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B01HHURN3W,3.0,1408995743000,2014-08-25 19:42:23,928257,166566
2,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B00L0YLRUW,1.0,1439226089000,2015-08-10 17:01:29,928257,109831
3,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B017T99JPG,5.0,1456772365000,2016-02-29 18:59:25,928257,148767
4,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B01LW71IBJ,5.0,1456772571000,2016-02-29 19:02:51,928257,173422


In [6]:
print("Min datetime:",reviews['datetime'].min())
print("Max datetime",reviews['datetime'].max())

Min datetime: 1999-06-13 22:10:04
Max datetime 2023-09-12 09:00:55.248000


In [7]:
reviews.dtypes

,0
user_id,object
parent_asin,object
rating,float64
timestamp,int64
datetime,datetime64[ns]
user_idx,int64
item_idx,int64


In [8]:
reviews = reviews.sort_values("datetime",ascending=True).reset_index(drop=True)


In [9]:
print(reviews['datetime'].head())
print()
print(reviews['datetime'].tail())

0   1999-06-13 22:10:04
1   1999-07-13 15:29:53
2   1999-07-13 17:18:51
3   1999-07-14 04:51:14
4   1999-07-14 20:53:38
Name: datetime, dtype: datetime64[ns]

15473531   2023-09-10 22:24:01.219
15473532   2023-09-10 23:04:26.164
15473533   2023-09-11 02:17:59.206
15473534   2023-09-11 20:07:31.278
15473535   2023-09-12 09:00:55.248
Name: datetime, dtype: datetime64[ns]


In [10]:
n = len(reviews)

train_end = int( n*0.80)
val_end = int(n*0.90)
print("Train-End",train_end)
print("Validation-End",val_end)

Train-End 12378828
Validation-End 13926182


In [11]:
train = reviews.iloc[:train_end]
val = reviews.iloc[train_end:val_end]
test = reviews.iloc[val_end:]

In [12]:
15473535*0.80

12378828.0

In [13]:
# Check the date range of each split
print("Train:", train["datetime"].min(), "->", train["datetime"].max())
print("Validation:", val["datetime"].min(), "->", val["datetime"].max())
print("Test:", test["datetime"].min(), "->", test["datetime"].max())

Train: 1999-06-13 22:10:04 -> 2021-03-02 23:31:55.579000
Validation: 2021-03-02 23:32:07.332000 -> 2022-04-03 17:48:52.987000
Test: 2022-04-03 17:48:53.088000 -> 2023-09-12 09:00:55.248000


In [14]:
# Check the number of interactions in each split
print("Train:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

Train: (12378828, 7)
Validation: (1547354, 7)
Test: (1547354, 7)


In [15]:
# Check for users in validation that are not present in training
train_users = set(train['user_idx'])
val_users = set(val['user_idx'])

new_val_user = val_users - train_users

In [16]:
test_users = set(test['user_idx'])

new_test_users = test_users - train_users

In [17]:
train.head()

,user_id,parent_asin,rating,timestamp,datetime,user_idx,item_idx
0,AGUH4HQWSFAZEZWWJUPENAITOIAQ,B00000JBAT,5.0,929311804000,1999-06-13 22:10:04,1158914,301
1,AGH2EPQBBP4B6MQALE3IPQ3NGA6A,B00000JFIF,5.0,931879793000,1999-07-13 15:29:53,986979,327
2,AG4ZJVVMOHDHXHN3WXQWNZU3NNPQ,B00000JBAT,2.0,931886331000,1999-07-13 17:18:51,858313,301
3,AEUVZ6UTQAPJNFDG3JA2KMGOO32A,B00000J40U,5.0,931927874000,1999-07-14 04:51:14,345430,275
4,AHCXZZ6R4US43PBVXXVN47OTTVHQ,B00000J4FY,5.0,931985618000,1999-07-14 20:53:38,1344817,282


In [18]:
train_items = set(train['item_idx'])
val_items = set(val['item_idx'])
test_items = set(test['item_idx'])


new_val_items = val_items - train_items
new_test_items = test_items - train_items
